In [1]:
from kiteconnect import KiteConnect

API_KEY = "yidl20ioz7kuy7yv"
API_SECRET = "luhm43fvkvr68hr9dzl6hejkuh3r7pqz"

kite = KiteConnect(api_key=API_KEY)

# Step 1: Open this URL in browser
print("Open this URL in your browser:")
print(kite.login_url())
print()



Open this URL in your browser:
https://kite.zerodha.com/connect/login?api_key=yidl20ioz7kuy7yv&v=3



In [3]:
# Step 2: After login, you'll be redirected to something like:
# https://127.0.0.1?request_token=xxxxxxxx&action=login&status=success
# Copy the request_token value and paste below

request_token = input("BhWtvO1vJGpqxX2I5kldI8QPmySeBQXW").strip()

session = kite.generate_session(request_token, api_secret=API_SECRET)


InputException: `request_token` should be minimum 10 characters in length.

In [ ]:
access_token = session["access_token"]

print(f"\nAccess Token: {access_token}")

# Save to file so other scripts can use it
with open("access_token.txt", "w") as f:
    f.write(access_token)

print("Saved to access_token.txt")

In [ ]:
from kiteconnect import KiteConnect
import pandas as pd
import datetime
import os

API_KEY = "your_api_key_here"

# Load today's access token
with open("access_token.txt") as f:
    ACCESS_TOKEN = f.read().strip()

kite = KiteConnect(api_key=API_KEY)
kite.set_access_token(ACCESS_TOKEN)

# ── Instrument lookup (cache it to avoid repeated API calls) ──────────────────
def load_instruments(exchange="NSE"):
    cache_file = f"instruments_{exchange}.csv"
    # Refresh once per day
    if os.path.exists(cache_file):
        saved_date = datetime.date.fromtimestamp(os.path.getmtime(cache_file))
        if saved_date == datetime.date.today():
            return pd.read_csv(cache_file)

    df = pd.DataFrame(kite.instruments(exchange))
    df.to_csv(cache_file, index=False)
    return df

def get_token(symbol, exchange="NSE"):
    instruments = load_instruments(exchange)
    row = instruments[instruments["tradingsymbol"] == symbol]
    if row.empty:
        raise ValueError(f"Symbol {symbol} not found")
    return int(row.iloc[0]["instrument_token"])

# ── Fetch 15min candles (handles >60 day chunks automatically) ────────────────
def fetch_15min(symbol, from_date, to_date, exchange="NSE"):
    token = get_token(symbol, exchange)
    all_data = []
    current = from_date

    while current < to_date:
        chunk_end = min(current + datetime.timedelta(days=59), to_date)
        print(f"  Fetching {symbol}: {current} → {chunk_end}")
        candles = kite.historical_data(
            instrument_token=token,
            from_date=current,
            to_date=chunk_end,
            interval="15minute"
        )
        all_data.extend(candles)
        current = chunk_end + datetime.timedelta(days=1)

    df = pd.DataFrame(all_data)
    if df.empty:
        return df
    df.set_index("date", inplace=True)
    df.index = pd.to_datetime(df.index)
    return df[["open", "high", "low", "close", "volume"]]

# ── Run ───────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    symbols = ["RELIANCE", "TCS", "INFY", "HDFCBANK"]  # Add your symbols
    from_date = datetime.date(2024, 1, 1)
    to_date   = datetime.date(2025, 1, 1)

    os.makedirs("data", exist_ok=True)

    for sym in symbols:
        print(f"\nDownloading {sym}...")
        try:
            df = fetch_15min(sym, from_date, to_date)
            out_path = f"data/{sym}_15min.csv"
            df.to_csv(out_path)
            print(f"  Saved {len(df)} candles → {out_path}")
        except Exception as e:
            print(f"  Error: {e}")